# 04. EOT 시험 분석용 표본 추출: 화장지 (2026-10-06)

**목적**: 감정·원인(trigger) 추출(Attri et al., 2025, *Findings of EMNLP*)을 시험해 볼 리뷰 표본을 뽑는다.
- **급증 구간**: 화장지 검색이 급증한 2020년 3월 (코로나 사재기)
- **기준 구간**: 1년 전(2019년) 같은 주
- 구간마다 100건, 모두 200건. 이 중 50건은 팀원 2명이 직접 라벨을 붙여 모델 결과를 검증한다.

**논문 표본 추출 방식과 이 노트북** (쪽수는 Findings of EMNLP 2025 기준, p.5519)
| 논문 | 이 노트북 |
|---|---|
| 영역마다 **상품을 먼저** 무작위 비복원추출(SRSWOR)로 40개 뽑음 | 구간마다 상품 10개를 비복원추출로 뽑음 (100건 ÷ 상품당 10건) |
| 리뷰 길이 10~100단어만 사용 | `in_len_range` (03 노트북) 그대로 사용 |
| 상품마다 리뷰 10개를 **시기별 층화**로 뽑음 | 상품마다 리뷰 10개를 **주(week)별로 고르게** 뽑음 (아래 4절) |
| 평점·도움돼요 수는 모델 입력에서 제외 | 표본 파일에는 분석용으로 남기지만, 라벨 시트·모델 입력에는 본문만 넣음 |
| 중복 제거 | `dup_same_user_text`(03 노트북) 표시된 리뷰 제외 |
| 영어 리뷰 | 후보 리뷰에만 언어 판별기(lingua)를 돌려 영어만 남김 |

**입력**: `data/interim/text/reviews_text.parquet` (03 노트북 결과)
**출력** (`data/processed/eot_pilot/`, 작아서 git에 올린다)
- `eot_pilot_sample.csv`: 표본 200건
- `label_sheet_A.csv`, `label_sheet_B.csv`: 사람 라벨용 50건 (라벨러 2명이 각자 하나씩)

**실행**: VS Code에서 열고 커널 `.venv` 확인 → **Run All** → 끝나면 **Ctrl+S**. **[점검]** 표시 출력을 Claude에게 보내서 확인받는다.

## 1. 준비

- `lingua-language-detector`: 언어 판별 패키지. 짧은 글에서도 정확한 편이라 리뷰에 맞다. 처음 한 번만 설치한다 (설치 후 커널 재시작 필요 없음).
- 아래 **설정값**만 바꾸면 구간·표본 크기를 바꿔 다시 뽑을 수 있다.
  - `SPIKE_START`, `SPIKE_END`: 급증 구간. 서연님의 Trends spike 결과(`spikes.csv`)가 나오기 전이라 **임시로 2020-03-08 ~ 2020-04-04 (일요일 시작 4주)** 로 둔다. 결과가 나오면 이 두 줄만 고친다.
  - 기준 구간은 급증 구간을 **정확히 52주(364일) 앞으로** 옮겨서 자동으로 정한다. 요일이 맞아서 주 단위 비교가 깔끔하다 (2019-03-10 ~ 2019-04-06).
  - `SEED`: 난수 시드. 같은 시드면 누가 돌려도 같은 표본이 나온다 (재현성).

In [1]:
%pip install -q lingua-language-detector

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 150)

# ---- 설정값 ----
PRODUCT        = "TOILET_PAPER"
SPIKE_START    = "2020-03-08"   # 임시. 서연님 spikes.csv 나오면 교체
SPIKE_END      = "2020-04-04"   # 이 날짜 포함
N_PER_PERIOD   = 100            # 구간별 표본 크기
N_PRODUCTS     = 10             # 구간별로 뽑을 상품 수
PER_PRODUCT    = 10             # 상품당 리뷰 수 (논문과 같음)
N_LABEL        = 50             # 사람 라벨 건수 (구간별 절반씩)
SEED           = 20261006
# ----------------

spike_start = pd.Timestamp(SPIKE_START); spike_end = pd.Timestamp(SPIKE_END)
base_start  = spike_start - pd.Timedelta(days=364); base_end = spike_end - pd.Timedelta(days=364)
print("급증 구간:", spike_start.date(), "~", spike_end.date())
print("기준 구간:", base_start.date(), "~", base_end.date())

DATA = Path("../data")
IN_PATH = DATA / "interim/text/reviews_text.parquet"
OUT_DIR = DATA / "processed/eot_pilot"
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.sql("SET TimeZone = 'UTC'")
try:
    con.sql("SET enable_progress_bar = false")   # VS Code 위젯 경고창 방지
except Exception:
    pass
rng = np.random.default_rng(SEED)

급증 구간: 2020-03-08 ~ 2020-04-04
기준 구간: 2019-03-10 ~ 2019-04-06


## 2. 후보 리뷰 모으기

두 구간에 쓰인 화장지 리뷰 중 아래 조건을 모두 만족하는 리뷰가 후보다.
- 본문이 비어 있지 않음 (`is_empty` = False)
- 10~100단어 (`in_len_range` = True, 논문 조건)
- 같은 사람·같은 본문 중복이 아님 (`dup_same_user_text` = False)
- 비ASCII 글자가 많지 않음 (`high_non_ascii` = False)

`ORDER BY review_id`: 파일 안 행 순서와 상관없이 항상 같은 순서로 읽어서, 시드가 같으면 표본도 같게 만든다.

In [3]:
pool = con.sql(f"""
    SELECT review_id, asin, product_type, week, review_date, text_clean, n_words, rating, helpful_vote,
           CASE WHEN review_date BETWEEN DATE '{spike_start.date()}' AND DATE '{spike_end.date()}' THEN 'spike'
                ELSE 'baseline' END AS period
    FROM read_parquet('{IN_PATH.as_posix()}')
    WHERE product_type = '{PRODUCT}'
      AND (review_date BETWEEN DATE '{spike_start.date()}' AND DATE '{spike_end.date()}'
           OR review_date BETWEEN DATE '{base_start.date()}' AND DATE '{base_end.date()}')
      AND NOT is_empty AND in_len_range AND NOT dup_same_user_text AND NOT high_non_ascii
    ORDER BY review_id
""").df()
pool["week"] = pd.to_datetime(pool["week"]).dt.date
print("후보 리뷰 수:", len(pool))
pool.groupby("period").size()

후보 리뷰 수: 2522


period
baseline     673
spike       1849
dtype: int64

## 3. [점검] 언어 판별 (후보 리뷰에만)

03 노트북에서 말한 대로, 110만 건 전체가 아니라 **여기 후보 리뷰에만** 언어 판별을 한다.
- 판별 대상 언어를 Amazon 리뷰에 나올 만한 주요 언어로 제한한다. 전체 75개 언어를 다 불러오면 메모리를 많이 쓰고, 짧은 영어 글이 드문 언어로 잘못 잡히기도 한다.
- 영어로 판별된 리뷰만 남긴다.
- 아래 표는 **영어가 아니라고 판별된 리뷰**다. 실제로는 영어인데 잘못 빠진 리뷰가 많으면 알려 준다 (그러면 기준을 조정한다).

In [4]:
from lingua import Language, LanguageDetectorBuilder
LANGS = [Language.ENGLISH, Language.SPANISH, Language.FRENCH, Language.GERMAN, Language.PORTUGUESE,
         Language.ITALIAN, Language.DUTCH, Language.CHINESE, Language.JAPANESE, Language.KOREAN,
         Language.ARABIC, Language.RUSSIAN, Language.HINDI]
detector = LanguageDetectorBuilder.from_languages(*LANGS).build()

pool["lang"] = [ (lambda l: l.name.lower() if l else "unknown")(detector.detect_language_of(t)) for t in pool["text_clean"] ]
print(pool.groupby(["period", "lang"]).size().unstack(fill_value=0))
pool.loc[pool["lang"] != "english", ["period", "lang", "text_clean"]].head(15)

lang      dutch  english  spanish
period                           
baseline      0      671        2
spike         1     1841        7


,period,lang,text_clean
609,baseline,spanish,Estan muy chiquitos los rollos de papel y delgadita las hojas del papel
673,spike,spanish,Nesesito más papel para mi uso nada más yo siempre copro para mi
1799,spike,spanish,Me gusta tener productos básicos en casa. ¡Gracias! (I like it when I have enough supplies at home. Thanks!)
1844,baseline,spanish,Bueno pero ya lo pusieron muy caro y ya compre barios aun no se cual es el regalo
2036,spike,spanish,Muy buena calidad no se desbarata o se rompe es fuerte
2040,spike,spanish,"El papel ingenico más corriente que e mirado, tocado y usado. Pero para este tiempo de crisis con el corona virus que se puede hacer?"
2172,spike,dutch,Súper soft and it didn’t hurt very good and nice.
2242,spike,spanish,"Que llego y que tengo stock, para estos tiempos dificiles., y que tengo para compartir con alguien mas.Gracias"
2310,spike,spanish,En lo personal te limpia las nachas y esta suave no hay nada que discutir
2455,spike,spanish,"no me agrado que decia en el mensaje que me enviaron que eran big rollos y son lo mas pequeños que he visto en los ultimos tiempos,ya esos no se v..."


In [5]:
cand = pool[pool["lang"] == "english"].reset_index(drop=True)
print("영어 후보 리뷰 수:")
cand.groupby("period").size()

영어 후보 리뷰 수:


period
baseline     671
spike       1841
dtype: int64

## 4. 표본 추출 (논문 방식: 상품 먼저 → 상품 안에서 시기별 층화)

구간마다 아래를 한다.
1. **상품 뽑기**: 그 구간에 후보 리뷰가 `PER_PRODUCT`(10)건 이상 있는 상품 중에서 `N_PRODUCTS`(10)개를 **비복원 단순무작위추출(SRSWOR)** 로 뽑는다. 논문이 상품 40개를 먼저 뽑은 것과 같은 방식이다.
   - 10건 이상인 상품이 10개보다 적으면, 모자란 만큼을 리뷰가 적은 상품에서 무작위로 더 뽑아 100건을 채운다. 이 경우 경고를 출력한다.
2. **상품 안에서 리뷰 뽑기 (시기별 층화)**: 상품의 후보 리뷰를 주(week)별로 나눠 각 주 안에서 무작위로 섞은 뒤, **주를 돌아가며 한 건씩** 꺼내 10건을 채운다. 그러면 리뷰가 한 주에 몰리지 않고 구간 전체에 고르게 퍼진다. 논문의 "temporally stratified"를 4주 구간에 맞게 적용한 것이다.

`rng`(시드 고정 난수)만 쓰기 때문에 다시 돌려도 같은 표본이 나온다.

In [6]:
def stratified_by_week(df, k):
    """주별로 섞은 뒤 주를 돌아가며 한 건씩 꺼내 k건을 고른다."""
    weeks = sorted(df["week"].unique())
    queues = {w: list(rng.permutation(df.index[df["week"] == w])) for w in weeks}
    picked = []
    while len(picked) < k and any(queues.values()):
        for w in weeks:
            if queues[w] and len(picked) < k:
                picked.append(queues[w].pop())
    return df.loc[picked]

def sample_period(cand, period):
    c = cand[cand["period"] == period]
    counts = c.groupby("asin").size()
    big   = sorted(counts[counts >= PER_PRODUCT].index)
    small = sorted(counts[counts <  PER_PRODUCT].index)
    print(f"[{period}] 후보 {len(c)}건 / 상품 {len(counts)}개 / {PER_PRODUCT}건 이상 상품 {len(big)}개")

    chosen = list(rng.choice(big, size=min(N_PRODUCTS, len(big)), replace=False))
    parts = [stratified_by_week(c[c["asin"] == a], PER_PRODUCT) for a in chosen]
    n = sum(len(p) for p in parts)
    if n < N_PER_PERIOD:
        print(f"  경고: {PER_PRODUCT}건 이상 상품이 부족해 {N_PER_PERIOD - n}건을 리뷰가 적은 상품에서 채움")
        rest = [a for a in big if a not in chosen] + small
        for a in rng.permutation(rest):
            if n >= N_PER_PERIOD:
                break
            p = stratified_by_week(c[c["asin"] == a], min(PER_PRODUCT, N_PER_PERIOD - n))
            parts.append(p); n += len(p)
    out = pd.concat(parts)
    if len(out) < N_PER_PERIOD:
        print(f"  경고: 후보가 부족해 {len(out)}건만 뽑힘")
    return out

sample = pd.concat([sample_period(cand, "spike"), sample_period(cand, "baseline")]).reset_index(drop=True)
print("\n표본 크기:", len(sample), "/ review_id 중복:", sample["review_id"].duplicated().sum(), "(0이어야 함)")

[spike] 후보 1841건 / 상품 159개 / 10건 이상 상품 55개
[baseline] 후보 671건 / 상품 92개 / 10건 이상 상품 18개

표본 크기: 200 / review_id 중복: 0 (0이어야 함)


## 5. [점검] 표본 요약

- 구간별 건수·상품 수·단어 수 중앙값
- 평균 평점은 **모델에 넣지 않고** 두 구간이 얼마나 다른지 참고로만 본다
- 주별 건수: 리뷰가 구간 안에 고르게 퍼졌는지 확인한다

In [7]:
display(sample.groupby("period").agg(리뷰=("review_id", "size"), 상품=("asin", "nunique"),
                                      단어수_중앙값=("n_words", "median"), 평균평점=("rating", "mean")).round(2))
display(sample.pivot_table(index="week", columns="period", values="review_id", aggfunc="size", fill_value=0))
sample.groupby(["period", "asin"]).size().unstack(0, fill_value=0)

,리뷰,상품,단어수_중앙값,평균평점
period,,,,
baseline,100,10,17.0,3.89
spike,100,10,22.0,3.61


period,baseline,spike
week,,
2019-03-10,30,0
2019-03-17,28,0
2019-03-24,23,0
2019-03-31,19,0
2020-03-08,0,24
2020-03-15,0,31
2020-03-22,0,23
2020-03-29,0,22


period,baseline,spike
asin,,
B008I7TNDW,10,0
B009R6725E,0,10
B01AZ15E22,0,10
B01M9I0F3F,0,10
B074CR89PR,10,0
B074CTMZ3Y,10,0
B074MJLYNZ,10,0
B0795VWV8K,10,0
B0795XF9HY,10,0


## 6. [점검] 표본 예시

구간별로 5건씩 본문을 본다. 화장지 리뷰가 맞는지, 감정이 드러나는 글이 섞여 있는지 눈으로 확인한다.

In [8]:
sample.groupby("period").head(5)[["period", "week", "rating", "text_clean"]]

,period,week,rating,text_clean
0,spike,2020-03-15,3.0,"Knock off 2 ply, not bad but not the real thing. I use this brand and type specifically and this is a knock off."
1,spike,2020-03-22,1.0,"Only 143 sheets, not the 176 they claim.... little itty bitty rolls of toilet paper"
2,spike,2020-03-29,1.0,I ordered this through Amazon Fresh without reading the reviews - so annoyed with these TP. They are tiny - super small rolls - waste of money. Pl...
3,spike,2020-03-15,1.0,These are like no double rolls I’ve ever seen. More like a mini rolls. I am very disappointed and feel very ripped off.
4,spike,2020-03-22,1.0,"Received the package of 12 double rolls, appears to be 12 single rolls in a package that says double. DO NOT BUY."
100,baseline,2019-03-10,2.0,"I typically buy my TP at Costco. I big package of mega rolls for only 15.99, This looked similar online. But it arrived yesterday and was clearly ..."
101,baseline,2019-03-17,5.0,Quality & price is very comparable to Angel Soft. Minimal lint left behind (ahhemm). Works well for me. Not sure why the negative reviews. If you'...
102,baseline,2019-03-24,4.0,You get a large amount for what you pay and the 2-ply is great. I am not looking to pay a ton for really nice quality toilet paper but I still wan...
103,baseline,2019-03-31,5.0,Good value for money! I was surprised how much came in the box.
104,baseline,2019-03-10,5.0,Lasted all winter only had to buy toilet paper as of last month..


## 7. 저장: 표본 파일

`eot_pilot_sample.csv`는 다음 단계(Colab에서 Mistral-7B 실행)의 입력이다. 모델에는 `review_id`와 `text_clean`만 넣는다. 나머지 열은 결과를 구간·평점과 엮어 분석할 때 쓴다.
- `utf-8-sig`: 엑셀에서 열어도 글자가 깨지지 않게 하는 인코딩

In [9]:
cols = ["review_id", "period", "asin", "product_type", "week", "review_date", "rating", "helpful_vote", "n_words", "lang", "text_clean"]
sample = sample.sort_values(["period", "asin", "review_date", "review_id"])[cols]
sample.to_csv(OUT_DIR / "eot_pilot_sample.csv", index=False, encoding="utf-8-sig")
print("저장:", OUT_DIR / "eot_pilot_sample.csv", len(sample), "건")

저장: ..\data\processed\eot_pilot\eot_pilot_sample.csv 200 건


## 8. 저장: 사람 라벨 시트 (50건)

모델 결과를 검증하려면 사람이 붙인 정답이 필요하다. 논문은 전문가 3명이 라벨을 붙여 일치도(Fleiss κ 감정 0.88)를 보고했다. 우리는 **팀원 2명이 같은 50건을 각자** 라벨링해서 두 사람 일치도(Cohen κ)와 모델 정확도(P/R/F1)를 계산한다.

- 표본에서 구간별로 25건씩 무작위로 뽑아 순서를 섞는다.
- **구간·평점·상품은 시트에 넣지 않는다.** 라벨러가 "사재기 시기니까 불안이겠지" 하고 짐작하지 않게 하려는 것이다 (논문이 평점을 뺀 것과 같은 이유).
- 열 구성 (논문 부록 D.8 출력 형식에 맞춤)
  - `Joy` ~ `Anticipation` (Plutchik 8감정): 그 감정이 있으면 **원인이 되는 구절을 본문에서 그대로 복사**해 넣는다. 여러 개면 ` | `로 구분한다. 감정이 없으면 비워 둔다.
  - `Neutral`: 감정이 하나도 없으면 1
  - `memo`: 애매한 점 메모
- 라벨링 지침(논문 부록 D)은 다음 단계에서 프롬프트와 함께 한국어로 정리해 준다. **지침을 받기 전에는 라벨링을 시작하지 않는다.**

In [10]:
EMOTIONS = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation"]

lab = pd.concat([sample[sample["period"] == p].sample(n=min(N_LABEL // 2, (sample["period"] == p).sum()),
                                                       random_state=SEED) for p in ["spike", "baseline"]])
lab = lab.sample(frac=1, random_state=SEED).reset_index(drop=True)
sheet = pd.DataFrame({"no": range(1, len(lab) + 1), "review_id": lab["review_id"], "text": lab["text_clean"]})
for e in EMOTIONS + ["Neutral", "memo"]:
    sheet[e] = ""
for who in ["A", "B"]:
    sheet.to_csv(OUT_DIR / f"label_sheet_{who}.csv", index=False, encoding="utf-8-sig")
print("라벨 시트:", len(sheet), "건 / 구간별:", lab["period"].value_counts().to_dict())
sheet.head(3)

라벨 시트: 50 건 / 구간별: {'baseline': 25, 'spike': 25}


,no,review_id,text,Joy,Trust,Fear,Surprise,Sadness,Disgust,Anger,Anticipation,Neutral,memo
0,1,51574da5c51f31a8,It gets the job done. Nothing fancy. I like ordering online because I don't have to go to the store. This large package lasts us a long time as we...,,,,,,,,,,
1,2,260a216ad687a8e7,Plenty of paper but the thinnest two ply toilet paper I’ve ever used. It cancels out the bargain of buying large amount.,,,,,,,,,,
2,3,993afaa9b631f9ab,"I liked the packaging, but When I went to use it it was like a very poor recycled type of paper that just disintegrated in your hands not the qual...",,,,,,,,,,


## 9. 로그에 붙일 문장

아래 출력을 복사해 `logs/preprocessing_log.md` 맨 아래에 붙인다.

In [11]:
by = sample.groupby("period").agg(n=("review_id", "size"), p=("asin", "nunique"))
pc = pool.groupby("period").size(); cc = cand.groupby("period").size()
lines = [
    "### 2026-10-06 T2. EOT 시험 표본 추출: 화장지 (김지우)",
    "노트북: `notebooks/04_eot_pilot_sample.ipynb`. 기준: Attri et al. (2025) 표본 추출 (p.5519)",
    "",
    f"- 급증 구간 {spike_start.date()} ~ {spike_end.date()} (임시, Trends spike 결과로 교체 예정) / 기준 구간 {base_start.date()} ~ {base_end.date()} (52주 전)",
    f"- 후보 조건: 10~100단어, 빈 본문·중복·비ASCII 많음 제외 → 급증 {pc.get('spike', 0):,}건 / 기준 {pc.get('baseline', 0):,}건",
    f"- 언어 판별(lingua) 후 영어만: 급증 {cc.get('spike', 0):,}건 / 기준 {cc.get('baseline', 0):,}건",
    f"- 추출: 구간마다 상품 {N_PRODUCTS}개 비복원추출 → 상품당 {PER_PRODUCT}건 주별 층화. 시드 {SEED}",
    f"- 결과: 급증 {by.loc['spike', 'n']}건(상품 {by.loc['spike', 'p']}개) / 기준 {by.loc['baseline', 'n']}건(상품 {by.loc['baseline', 'p']}개) → `data/processed/eot_pilot/eot_pilot_sample.csv`",
    f"- 사람 라벨용 {len(sheet)}건 (구간별 {N_LABEL // 2}건, 구간·평점 숨김) → `label_sheet_A.csv`, `label_sheet_B.csv`",
]
print("\n".join(lines))

### 2026-10-06 T2. EOT 시험 표본 추출: 화장지 (김지우)
노트북: `notebooks/04_eot_pilot_sample.ipynb`. 기준: Attri et al. (2025) 표본 추출 (p.5519)

- 급증 구간 2020-03-08 ~ 2020-04-04 (임시, Trends spike 결과로 교체 예정) / 기준 구간 2019-03-10 ~ 2019-04-06 (52주 전)
- 후보 조건: 10~100단어, 빈 본문·중복·비ASCII 많음 제외 → 급증 1,849건 / 기준 673건
- 언어 판별(lingua) 후 영어만: 급증 1,841건 / 기준 671건
- 추출: 구간마다 상품 10개 비복원추출 → 상품당 10건 주별 층화. 시드 20261006
- 결과: 급증 100건(상품 10개) / 기준 100건(상품 10개) → `data/processed/eot_pilot/eot_pilot_sample.csv`
- 사람 라벨용 50건 (구간별 25건, 구간·평점 숨김) → `label_sheet_A.csv`, `label_sheet_B.csv`
